In [ ]:
from requirements_deprecated import *

# 0. FCI, DFT

In [ ]:
import time
from pyscf import gto, scf, dft, fci

def Classical_CO2_Comparison(dist):
    
    atom_string = f"C 0.0 0.0 0.0; O 0.0 0.0 {dist}; O 0.0 0.0 {-dist}"
    
    mol = gto.M(
        atom=atom_string,
        basis="sto-3g",
        charge=0,
        spin=0,
        verbose=0 
    )
    
    metrics = {}
    
    hf_start = time.time()
    mf = scf.RHF(mol)
    energy_hf = mf.kernel()
    metrics['time_hf'] = time.time() - hf_start
    
    dft_start = time.time()
    mf_dft = dft.RKS(mol)
    mf_dft.xc = 'b3lyp'
    energy_dft = mf_dft.kernel()
    metrics['time_dft'] = time.time() - dft_start
    
    fci_start = time.time()
    myfci = fci.FCI(mf)
    energy_fci = myfci.kernel()[0]
    metrics['time_fci'] = time.time() - fci_start
    
    return energy_hf, energy_dft, energy_fci, metrics

In [ ]:
import numpy as np
from tqdm import tqdm

distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
distances = np.sort(distances)

results_classical = []

for d in tqdm(distances, desc="Bond Lengths"):
    e_hf, e_dft, e_fci, metrics = Classical_CO2_Comparison(d)
    
    data_point = {
        "distance": d,
        "energy_hf": e_hf,
        "energy_dft": e_dft,
        "energy_fci": e_fci,
        "metrics": metrics
    }
    results_classical.append(data_point)

save_data(results_classical, "co2_classical_benchmarks.pkl")

# 1. Active Space

In [ ]:
import time
import sys
import os
import numpy as np

root_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..'))
sys.path.append(root_dir)

from utils import save_data

from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_algorithms.optimizers import SLSQP
from qiskit_algorithms import VQE
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit import transpile

num_threads = int(sys.argv[1]) if len(sys.argv) > 1 else 42
os.environ["OMP_NUM_THREADS"] = str(num_threads)

def VQE_CO2_ActiveSpace_Aer(dist, num_electrons, num_spatial_orbitals):
    start_time = time.time()

    atom_string = f"C 0.0 0.0 0.0; O 0.0 0.0 {dist}; O 0.0 0.0 {-dist}"

    driver = PySCFDriver(
        atom=atom_string, 
        basis="sto-3g", 
        charge=0, 
        spin=0
    )
    
    problem = driver.run()
    
    transformer = ActiveSpaceTransformer(
        num_electrons=num_electrons, 
        num_spatial_orbitals=num_spatial_orbitals
    )
    reduced_problem = transformer.transform(problem)
    
    mapper = JordanWignerMapper()
    
    ansatz = UCCSD(
        reduced_problem.num_spatial_orbitals,
        reduced_problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            reduced_problem.num_spatial_orbitals,
            reduced_problem.num_particles,
            mapper,
        ),
    )
    
    transpiled_ansatz = transpile(ansatz, basis_gates=['u', 'cx'], optimization_level=1)
    
    estimator = AerEstimator(
            run_options={
                "shots": None, 
                "method": "statevector",
                "max_parallel_threads": num_threads
            },
            approximation=True
    )
    
    vqe = VQE(estimator, ansatz, SLSQP(maxiter=100))
    vqe.initial_point = [0.0] * ansatz.num_parameters
    
    solve_start = time.time()
    
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(reduced_problem)
    solve_elapsed = time.time() - solve_start
    
    total_elapsed = time.time() - start_time

    metrics = {}
    
    metrics['num_qubits'] = ansatz.num_qubits
    metrics['num_parameters'] = ansatz.num_parameters
    metrics['circuit_depth'] = transpiled_ansatz.depth()
    metrics['cnot_count'] = transpiled_ansatz.count_ops().get('cx', 0)
    
    metrics['solve_time'] = solve_elapsed
    metrics['total_time'] = total_elapsed
    
    if hasattr(result.raw_result, 'cost_function_evals'):
        metrics['cost_function_evals'] = result.raw_result.cost_function_evals
    
    if hasattr(result.raw_result, 'optimizer_result'):
        opt_res = result.raw_result.optimizer_result
        if hasattr(opt_res, 'nit'):
            metrics['nit'] = opt_res.nit
        if hasattr(opt_res, 'nfev'):
            metrics['nfev'] = opt_res.nfev
        if hasattr(opt_res, 'fun'):
            metrics['fun'] = opt_res.fun
            
    return result.total_energies[0], metrics


if __name__ == "__main__":
    print(f"--- Active Space comparison : {num_threads} cores ---")
    
    distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
    distances = np.sort(distances)

    ACTIVE_SPACES = [(2, 2), (4, 4), (6, 6), (8,8)]

    results = {f"CAS_{e}_{o}": [] for e, o in ACTIVE_SPACES}

    for num_elec, num_orbs in ACTIVE_SPACES:
        cas_label = f"CAS_{num_elec}_{num_orbs}"
        
        print(f"\nActive Space: {cas_label} ({num_orbs * 2} qubits)")
        
        for d in distances:
        
            loop_start = time.time()
            
            energy, metrics = VQE_CO2_ActiveSpace_Aer(d, num_elec, num_orbs)
            
            data_point = {
                "distance": d,
                "energy": energy,
                "metrics": metrics
            }
            results[cas_label].append(data_point)
            
            loop_time = time.time() - loop_start
            print(f"Dist: {d:.4f} Å | Time: {loop_time:.2f}s")
            
            save_data(results, "co2_active_space_comparison_Aer_2468.pkl")

    print("\n--- Done ---")

# 2. Tapered

In [ ]:
import time
import sys
import os
import numpy as np

root_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..'))
sys.path.append(root_dir)

from utils import save_data

from qiskit.compiler import transpile
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper, BravyiKitaevMapper, ParityMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

num_threads = int(sys.argv[1]) if len(sys.argv) > 1 else 8
os.environ["OMP_NUM_THREADS"] = str(num_threads)

def VQE_CO2_Tapered_Mapper_Aer(dist, num_electrons, num_spatial_orbitals, mapper_name):
    start_time = time.time()

    atom_string = f"C 0.0 0.0 0.0; O 0.0 0.0 {dist}; O 0.0 0.0 {-dist}"

    driver = PySCFDriver(
        atom=atom_string, 
        basis="sto-3g", 
        charge=0, 
        spin=0
    )
    
    problem = driver.run()
    
    transformer = ActiveSpaceTransformer(
        num_electrons=num_electrons, 
        num_spatial_orbitals=num_spatial_orbitals
    )
    reduced_problem = transformer.transform(problem)
    
    if mapper_name == "JordanWigner":
        base_mapper = JordanWignerMapper()
    elif mapper_name == "BravyiKitaev":
        base_mapper = BravyiKitaevMapper()
    elif mapper_name == "Parity":
        base_mapper = ParityMapper(num_particles=reduced_problem.num_particles)
    else:
        raise ValueError("Invalid mapper name provided.")
        
    mapper = reduced_problem.get_tapered_mapper(base_mapper)
    
    ansatz = UCCSD(
        reduced_problem.num_spatial_orbitals,
        reduced_problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            reduced_problem.num_spatial_orbitals,
            reduced_problem.num_particles,
            mapper,
        ),
    )
    
    transpiled_ansatz = transpile(ansatz, basis_gates=['u', 'cx'], optimization_level=1)
    
    estimator = AerEstimator(
            run_options={
                "shots": None, 
                "method": "statevector",
                "max_parallel_threads": num_threads
            },
            approximation=True
    )
    
    vqe = VQE(estimator, ansatz, SLSQP(maxiter=100))
    vqe.initial_point = [0.0] * ansatz.num_parameters
    
    solve_start = time.time()
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(reduced_problem)
    solve_elapsed = time.time() - solve_start
    
    total_elapsed = time.time() - start_time

    metrics = {}
    
    metrics['num_qubits'] = ansatz.num_qubits
    metrics['num_parameters'] = ansatz.num_parameters
    metrics['circuit_depth'] = transpiled_ansatz.depth()
    metrics['cnot_count'] = transpiled_ansatz.count_ops().get('cx', 0)
    
    metrics['solve_time'] = solve_elapsed
    metrics['total_time'] = total_elapsed
    
    if hasattr(result.raw_result, 'cost_function_evals'):
        metrics['cost_function_evals'] = result.raw_result.cost_function_evals
    if hasattr(result.raw_result, 'optimizer_result'):
        opt_res = result.raw_result.optimizer_result
        if hasattr(opt_res, 'nit'):
            metrics['nit'] = opt_res.nit
        if hasattr(opt_res, 'nfev'):
            metrics['nfev'] = opt_res.nfev
        if hasattr(opt_res, 'fun'):
            metrics['fun'] = opt_res.fun
            
    return result.total_energies[0], metrics

if __name__ == "__main__":

    distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
    distances = np.sort(distances)

    MAPPERS = ["JordanWigner", "BravyiKitaev", "Parity"]
    ACTIVE_SPACES = [(2, 2), (4, 4), (6, 6)]

    results_tapered = {
        f"Tapered_{mapper}_CAS_{e}_{o}": [] 
        for mapper in MAPPERS 
        for e, o in ACTIVE_SPACES
    }

    print(f"--- Tapered Mapper and Active Space comparison : {num_threads} cores ---")

    for mapper_name in MAPPERS:
        for num_elec, num_orbs in ACTIVE_SPACES:
            
            label = f"Tapered_{mapper_name}_CAS_{num_elec}_{num_orbs}"
            
            print(f"\nEvaluating Configuration: {label}")
            
            for d in distances:
                
                print(f"Bond length: {d:.4f} Å")
                
                energy, metrics = VQE_CO2_Tapered_Mapper_Aer(
                    dist=d, 
                    num_electrons=num_elec, 
                    num_spatial_orbitals=num_orbs, 
                    mapper_name=mapper_name
                )
                
                data_point = {
                    "distance": d,
                    "energy": energy,
                    "metrics": metrics
                }
                results_tapered[label].append(data_point)

    save_data(results_tapered, "Tapered_mapper_comparison.pkl")
    print("\n--- Done ---")

# 3. Mapper

In [ ]:
import time
import sys
import os
import numpy as np

root_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..'))
sys.path.append(root_dir)

from utils import save_data

from qiskit.compiler import transpile
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper, BravyiKitaevMapper, ParityMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit_nature.second_q.algorithms import GroundStateEigensolver

num_threads = int(sys.argv[1]) if len(sys.argv) > 1 else 8
os.environ["OMP_NUM_THREADS"] = str(num_threads)

def VQE_CO2_Mapper_Comparison_Aer(dist, num_electrons, num_spatial_orbitals, mapper_name):

    start_time = time.time()

    atom_string = f"C 0.0 0.0 0.0; O 0.0 0.0 {dist}; O 0.0 0.0 {-dist}"

    driver = PySCFDriver(
        atom=atom_string, 
        basis="sto-3g", 
        charge=0, 
        spin=0
    )
    
    problem = driver.run()
    
    transformer = ActiveSpaceTransformer(
        num_electrons=num_electrons, 
        num_spatial_orbitals=num_spatial_orbitals
    )
    reduced_problem = transformer.transform(problem)
    
    if mapper_name == "JordanWigner":
        mapper = JordanWignerMapper()
    elif mapper_name == "BravyiKitaev":
        mapper = BravyiKitaevMapper()
    elif mapper_name == "Parity":
        mapper = ParityMapper(num_particles=reduced_problem.num_particles)
    else:
        raise ValueError("Invalid mapper name provided.")
    
    ansatz = UCCSD(
        reduced_problem.num_spatial_orbitals,
        reduced_problem.num_particles,
        mapper,
        initial_state=HartreeFock(
            reduced_problem.num_spatial_orbitals,
            reduced_problem.num_particles,
            mapper,
        ),
    )
    
    transpiled_ansatz = transpile(ansatz, basis_gates=['u', 'cx'], optimization_level=1)
    
    estimator = AerEstimator(
            run_options={
                "shots": None, 
                "method": "statevector",
                "max_parallel_threads": num_threads
            },
            approximation=True
    )
    
    vqe = VQE(estimator, ansatz, SLSQP(maxiter=100))
    vqe.initial_point = [0.0] * ansatz.num_parameters
    
    solve_start = time.time()
    calc = GroundStateEigensolver(qubit_mapper=mapper, solver=vqe)
    result = calc.solve(reduced_problem)
    solve_elapsed = time.time() - solve_start
    
    total_elapsed = time.time() - start_time

    metrics = {}
    
    metrics['num_qubits'] = ansatz.num_qubits
    metrics['num_parameters'] = ansatz.num_parameters
    metrics['circuit_depth'] = transpiled_ansatz.depth()
    metrics['cnot_count'] = transpiled_ansatz.count_ops().get('cx', 0)
    
    metrics['solve_time'] = solve_elapsed
    metrics['total_time'] = total_elapsed
    
    if hasattr(result.raw_result, 'cost_function_evals'):
        metrics['cost_function_evals'] = result.raw_result.cost_function_evals
    if hasattr(result.raw_result, 'optimizer_result'):
        opt_res = result.raw_result.optimizer_result
        if hasattr(opt_res, 'nit'):
            metrics['nit'] = opt_res.nit
        if hasattr(opt_res, 'nfev'):
            metrics['nfev'] = opt_res.nfev
        if hasattr(opt_res, 'fun'):
            metrics['fun'] = opt_res.fun
            
    return result.total_energies[0], metrics

if __name__ == "__main__":

    distances = np.concatenate([np.linspace(0.3, 1.0, 15), np.linspace(1.1, 2.5, 5)])
    distances = np.sort(distances)

    MAPPERS = ["JordanWigner", "BravyiKitaev", "Parity"]
    ACTIVE_SPACES = [(2, 2), (4, 4), (6, 6)]

    results = {
        f"{mapper}_CAS_{e}_{o}": [] 
        for mapper in MAPPERS 
        for e, o in ACTIVE_SPACES
    }

    print(f"--- Mappercomparison : {num_threads} cores ---")

    for mapper_name in MAPPERS:
        
        for num_elec, num_orbs in ACTIVE_SPACES:
        
            label = f"{mapper_name}_CAS_{num_elec}_{num_orbs}"
        
            print(f"\nEvaluating Configuration: {label}")
            
            for d in distances:
            
                print(f"Bond length: {d:.4f} Å")
            
                energy, metrics = VQE_CO2_Mapper_Comparison_Aer(
                    dist=d, 
                    num_electrons=num_elec, 
                    num_spatial_orbitals=num_orbs, 
                    mapper_name=mapper_name
                )
                
                data_point = {
                    "distance": d,
                    "energy": energy,
                    "metrics": metrics
                }
                results[label].append(data_point)

    save_data(results, "co2_mapper_matrix_comparison_Aer.pkl")
    print("\n-- Don ---")

# Test